### 遥感影像处理    
(1) 矢量裁剪    
(2) 光谱指数计算  
(3) 结果输出     

In [23]:
import rasterio as rio
from rasterio.warp import reproject, Resampling
from rasterio.transform import Affine


In [24]:
path_rsimg = 'data/rsimg/s2_20251206.tif'


##### 影像裁剪

In [25]:
## 空间范围裁剪
# 输入输出路径
path_clip_1 = 'data/rsimg/s2_20251206_clip1.tif'
# 定义裁剪范围（地理坐标：左、下、右、上）
clip_bounds_1 = (266675, 2733747, 270892, 2737296)  # 范围: (左, 下, 右, 上)
# 打开原始栅格
with rio.open(path_rsimg) as src:
    # 计算裁剪范围的像素窗口
    window = src.window(*clip_bounds_1)
    ## round_offsets():调整窗口的起始位置; round_shape():调整窗口的大小
    window = window.round_offsets().round_shape()  # 调整窗口，使其对齐到栅格数据 

    # 读取窗口内的数据, data为np.array数组
    data = src.read(window=window)
    # 计算裁剪后的新变换矩阵
    new_transform = src.window_transform(window)

    # 写入裁剪结果
    with rio.open(
            fp=path_clip_1,
            mode="w",
            driver="GTiff",
            height=window.height,
            width=window.width,
            count=src.count,
            dtype=data.dtype,
            crs=src.crs,
            transform=new_transform) as dst:
        dst.write(data) 



c:\Users\HP\miniconda3\envs\gis-course\lib\site-packages\rasterio\windows.py:729: RasterioDeprecationWarning: round_shape is deprecated and will be removed in Rasterio 2.0.0.
  warnings.warn(


In [26]:
## 基于矢量裁剪
import geopandas as gpd
from rasterio.mask import mask
path_ynu_vec = 'data/chenggong.gpkg'
path_rsimg_clip = 'data/rsimg/s2_20251206_clip2.tif'


In [27]:
chenggong_gpd = gpd.read_file(path_ynu_vec)
chenggong_polygons = chenggong_gpd.to_crs(epsg=32648).geometry.values

with rio.open(path_rsimg) as src:
    # 执行掩膜裁剪
    crop_data, crop_transform = mask(
        src,
        chenggong_polygons,
        crop=True,           # 裁剪到最小外接矩形
        all_touched=False,   # 只包含完全在内部的像素
        nodata=src.nodata    # 使用原始NoData值
    )

    # 更新元数据
    meta_ = src.meta.copy()
    meta_.update({
        'height': crop_data.shape[1],
        'width': crop_data.shape[2],
        'transform': crop_transform
    })

    with rio.open(path_rsimg_clip, 'w', **meta_) as dst:
        dst.write(crop_data)


#### 影像拼接

In [28]:
import rasterio
from rasterio.merge import merge

# 输入文件列表
path_rsimg_1 = 'data/rsimg/s2_20251206_clip1.tif'
path_rsimg_2 = 'data/rsimg/s2_20251206_clip2.tif'
path_mosaic = 'data/rsimg/s2_20251206_mosaic.tif'


In [29]:
# 读取所有栅格
src_1 = rio.open(path_rsimg_1)
src_2 = rio.open(path_rsimg_2)

# 执行拼接
merged_data, merged_transform = merge([src_1, src_2])

# 更新元数据
meta = src_1.meta.copy()
meta.update({
    "height": merged_data.shape[1],
    "width": merged_data.shape[2],
    "transform": merged_transform
    })
meta


{'driver': 'GTiff',
 'dtype': 'uint16',
 'nodata': None,
 'width': 669,
 'height': 1613,
 'count': 6,
 'crs': CRS.from_wkt('PROJCS["WGS 84 / UTM zone 48N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",105],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32648"]]'),
 'transform': Affine(20.0, 0.0, 266660.0,
        0.0, -20.0, 2765460.0)}

In [30]:
## 写入结果
with rasterio.open(fp=path_mosaic, 
                mode="w", 
                driver="GTiff",
                height=meta['height'],
                width=meta['width'],
                count=meta['count'],
                dtype=meta['dtype'],
                crs=meta['crs'],
                transform=meta['transform']) as dst:
    dst.write(merged_data)

# 关闭所有输入文件
src_1.close()
src_2.close() 

